### テキスト生成
+ デコード戦略(デコード手法, ハイパーパラメータ)がテキスト生成に影響する
+ 反復して文字生成をするので、分類系の1回の準伝播型よりも計算コストが高い
+ 自己回帰言語モデル, 因果的言語モデル

+ プロンプト $\boldsymbol{x} = x_1,x_2, \cdots, x_k$
+ テキストに出現するトークン列 $\boldsymbol{y} = y_1,y_2, \cdots, y_t$

確率モデル
$$P(y_1, \cdots, y_t | \boldsymbol{x}) = \prod_{t=1}^{N} P(y_t | y_{<t}, \boldsymbol{x})$$

言語モデルのヘッドは、各生成ステップで語彙中のトークンごとにロジット$z_{t,i}$を生成するので、softmaxを適用することにより、次に来そうなトークンの確率を得ることができる.
$$ P(y_t = w_i | y_{<t}, \boldsymbol{x}) = softmax(z_{t,i})$$

下記のトークン列を一気に求めるのは困難なので、近似してトークン列を求める
$$ \hat{\boldsymbol{y}} = argmax_{\boldsymbol{y}} P(\boldsymbol{y} | \boldsymbol{x})$$

### デコード戦略
+ 複数候補からの生成データの選び方: 貪欲法, ビームサーチ, ビームサーチ+Nグラムペナルティ
+ サンプリング手法: ランダムサンプリング
+ サンプリング対象の確率分布の形状を無理やり変更する(切り詰めたりなど。。。) : Top-kサンプリング, Top-pサンプリング

### 貪欲法によるデコード
+ 出力のロジット配列、もしくは確率配列から最も大きなインデックスに対応するIDを取り出して、逆マッピングして語彙を選ぶ
$$ \hat{y_t} = argmax_{y_{t}} P(y_t | y_{<t}, \boldsymbol{x})$$

In [1]:
import os
print("現在のHF_HOME:", os.environ.get("HF_HOME"))

print("\n--- transformersが実際に使うパスの確認 ---")
# デフォルトのキャッシュディレクトリを取得する関数を呼び出す
try:
    from transformers.utils import default_cache_path
    # 一般的なモデルのデフォルトキャッシュ先（huggingface/hub）を確認
    print("transformersのハブキャッシュ先:", os.path.join(os.environ.get("HF_HOME", "未設定"), "hub"))
except ImportError:
    # 古いバージョンの場合の互換性
    from transformers.file_utils import HF_MODULES_CACHE
    print("transformersのハブキャッシュ先 (旧):", HF_MODULES_CACHE)

現在のHF_HOME: X:\HuggingFaceCache

--- transformersが実際に使うパスの確認 ---
transformersのハブキャッシュ先 (旧): X:\HuggingFaceCache\modules


In [2]:
import os
# 環境変数を削除、または "0,1" を明示的に指定して制限を解除
if "CUDA_VISIBLE_DEVICES" in os.environ:
    del os.environ["CUDA_VISIBLE_DEVICES"]

In [3]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    counts = torch.cuda.device_count()
    for index in range(counts):
        print(f"cuda:{index}", torch.cuda.get_device_name(index))
    device = "cuda:0" # NVIDIA GeForce RTX 4080 SUPER
else:
    device = "cpu"

print(device)

cuda:0 NVIDIA GeForce RTX 4080 SUPER
cuda:1 NVIDIA GeForce RTX 4070
cuda:0


In [4]:
# 貪欲法を試す
from transformers import AutoTokenizer, AutoModelForCausalLM
model_name = 'gpt2-xl'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
C:\Users\InoueShinichi\.conda\envs\Book_Transformers\Lib\site-packages\transformers\modeling_utils.py:1435: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the 

すでに用意されている次の語彙を予測するgenerate()関数を独自実装する

In [5]:
import pandas as pd

input_txt = "Transformers are the"
input_ids = tokenizer(input_txt, return_tensors="pt")['input_ids'].to(device)
iterations = []
n_steps = 8 # 8時刻分語彙を生成する
choices_per_step = 5 # ロジット(確率)の上位5つ候補にする

with torch.no_grad():
    for _ in range(n_steps):
        iteration = dict()
        iteration['Input'] = tokenizer.decode(input_ids[0])
        output = model(input_ids=input_ids) # 順方向
        # 最初のバッチと最後のトークンのロジットを選択し、ソフトマックスを適用する
        next_token_logits = output.logits[0, -1, :]
        next_token_probs = torch.softmax(next_token_logits, dim=-1)
        sorted_ids = torch.argsort(next_token_probs, dim=-1, descending=True) # 降順
        print("sorted_ids", sorted_ids)
        # 最も高い確率のトークンを格納する
        for choice_idx in range(choices_per_step):
            token_id = sorted_ids[choice_idx]
            token_prob = next_token_probs[token_id].cpu().numpy()
            token_choice = (
                f"{tokenizer.decode(token_id)} ({100 * token_prob:.2f}%)"
            )
            iteration[f"Choice {choice_idx + 1}"] = token_choice
        # 予測した次のトークンを入力に追加する
        input_ids = torch.cat([input_ids, sorted_ids[None, 0, None]], dim=-1)
        iterations.append(iteration)

pd.DataFrame(iterations)

sorted_ids tensor([ 749,  691, 1266,  ...,  195,  208,  181], device='cuda:0')
sorted_ids tensor([2968, 3665, 2219,  ...,  212,  182,  205], device='cuda:0')
sorted_ids tensor([13373, 14958, 39185,  ...,   200,   208,   195], device='cuda:0')
sorted_ids tensor([1627,  287,  286,  ...,  200,  195,  194], device='cuda:0')
sorted_ids tensor([287, 286,  11,  ..., 200, 191, 195], device='cuda:0')
sorted_ids tensor([  262,  2106,  2253,  ...,   193,   195, 30905], device='cuda:0')
sorted_ids tensor([  995,  1578,  2106,  ...,   194, 30905,   195], device='cuda:0')
sorted_ids tensor([ 11,  13, 290,  ..., 191, 201, 200], device='cuda:0')


,Input,Choice 1,Choice 2,Choice 3,Choice 4,Choice 5
0,Transformers are the,most (8.53%),only (4.96%),best (4.65%),Transformers (4.37%),ultimate (2.16%)
1,Transformers are the most,popular (16.78%),powerful (5.37%),common (4.96%),famous (3.72%),successful (3.20%)
2,Transformers are the most popular,toy (10.63%),toys (7.23%),Transformers (6.60%),of (5.46%),and (3.76%)
3,Transformers are the most popular toy,line (34.38%),in (18.20%),of (11.71%),brand (6.10%),line (2.69%)
4,Transformers are the most popular toy line,in (46.28%),of (15.09%),", (4.94%)",on (4.40%),ever (2.72%)
5,Transformers are the most popular toy line in,the (65.99%),history (12.42%),America (6.91%),Japan (2.44%),North (1.40%)
6,Transformers are the most popular toy line in the,world (69.26%),United (4.55%),history (4.29%),US (4.23%),U (2.30%)
7,Transformers are the most popular toy line in ...,", (39.73%)",. (30.64%),and (9.87%),with (2.32%),today (1.74%)


generate()関数を貪欲法(max_new_tokens, do_sample=False)で使う

In [6]:
input_ids = tokenizer(input_txt, return_tensors="pt")['input_ids'].to(device)
output = model.generate(input_ids, max_new_tokens=n_steps, do_sample=False)
print(tokenizer.decode(output[0]))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Transformers are the most popular toy line in the world,


OpenAIのユニコーンの話が再現できるかチェック(max_lengthに大きな値を指定する)

In [7]:
max_length = 128
input_txt = """In a shocking finding, scientist descovered \
a herd of unicorns living in a remote, previously unexplored \
valley, in the Andes Mountains. Even more surprising to the \
researchers was the fact that the unicorns spoke perfect the English.\n\n
"""

input_ids = tokenizer(input_txt, return_tensors="pt")['input_ids'].to(device)
output_greedy = model.generate(input_ids, max_length=max_length, do_sample=False)
print(tokenizer.decode(output_greedy[0]))


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In a shocking finding, scientist descovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. Even more surprising to the researchers was the fact that the unicorns spoke perfect the English.


The researchers, from the University of California, Davis, and the University of Colorado, Boulder, were conducting a study on the Andean cloud forest, which is home to the rare species of cloud forest trees.


The researchers were surprised to find that the unicorns were able to communicate with each other, and even with humans.


The researchers were surprised to find that the unicorns


In [8]:
# # 貪欲法は生成タスクではほとんど使われない。算術計算のように確実に唯一解があるようなタスクで利用される。
# prompt = """
# 5 + 8 => 13
# 7 + 2 => 9
# 1 + 0 =>"""
#
# input_ids = tokenizer(prompt, return_tensors="pt")['input_ids'].to(device)
# output_greedy = model.generate(input_ids, max_length=max_length, do_sample=False)
# print(tokenizer.decode(output_greedy[0]))

### ビームサーチによるデコード
1. 各ステップで最も高い確率のトークンをデコードする代わりに、上位b件(ビーム数)の高い確率のトークンを記憶しておく。
2. 次の上位b件の集合(ビーム集合)は、既存の集合に接続しうるトークンのうち、最も可能性の高いb件を選択する。
+ 確率は確率の乗算によるコンピューターのアンダーフローを防ぐために、対数確率を使う.
$$logP(y_1, \cdots, y_t|\boldsymbol(x)) = \sum_{t=1}^{N}logP(y_t|y_{<t},\boldsymbol{x})$$

+ ビームサーチ法の欠点は、系列の部分部分で似たような出力を出すこと。

In [9]:
# e.g.
import numpy as np
print(0.5 ** 1024) # 確率の乗算
print(sum([np.log(0.5)] * 1024)) # 対数確率の加算

5.562684646268003e-309
-709.7827128933695


In [10]:
# ビームサーチ法でテキスト生成する
import torch.nn.functional as F

def log_probs_from_logits(logits, labels):
    logp = F.log_softmax(logits, dim=-1) # 正規化されていないロジットを正規化して対数確率として出力
    logp_label = torch.gather(logp, 2, labels.unsqueeze(2)).squeeze(-1)
    return logp_label

def sequence_logprob(modek, labels, input_len=0):
    with torch.no_grad():
        output = model(labels) # 順方向
        log_probs = log_probs_from_logits(output.logits[:, :-1, :], labels[:, 1:])
        seq_log_prob = torch.sum(log_probs[:, input_len:])
    return seq_log_prob.cpu().numpy()



OpenAIのプロンプト上で貪欲法によるデコーダの系列対数確率

In [11]:
# OpenAIのプロンプト上で貪欲法によるデコーダの系列対数確率
logp = sequence_logprob(model, output_greedy, input_len=len(input_ids[0]))
print(tokenizer.decode(output_greedy[0]))
print(f"\nlog-prob: {logp: .2f}")

In a shocking finding, scientist descovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. Even more surprising to the researchers was the fact that the unicorns spoke perfect the English.


The researchers, from the University of California, Davis, and the University of Colorado, Boulder, were conducting a study on the Andean cloud forest, which is home to the rare species of cloud forest trees.


The researchers were surprised to find that the unicorns were able to communicate with each other, and even with humans.


The researchers were surprised to find that the unicorns

log-prob: -87.77


ビームサーチで生成された系列

In [12]:
output_beam = model.generate(input_ids, max_length=max_length, do_sample=False, num_beams=5)
logp = sequence_logprob(model, output_beam, input_len=len(input_ids[0]))
print(tokenizer.decode(output_beam[0]))
print(f"\nlog-prob: {logp: .2f}")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In a shocking finding, scientist descovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. Even more surprising to the researchers was the fact that the unicorns spoke perfect the English.


The discovery of the unicorns was made by a team of scientists from the University of California, Davis, and the University of Colorado, Boulder.


The scientists were conducting a study of the Andes Mountains when they discovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. Even more surprising to the researchers was the fact that the unicorns

log-prob: -57.37


### ビームサーチ法 + nグラムペナルティ
+ ビームサーチ法の欠点である似たような出力を防ぐために, nグラムペナルティを加える
+ no_repeat_ngram_sizeパラメータを使う
+ dono
+ ｎグラムが出現したかを記憶しておき、以前に出現したnグラムを生成するならば、次のトークンの確率をゼロにする

In [13]:
output_beam_with_ngram = model.generate(input_ids, max_length=max_length, do_sample=False, num_beams=5, no_repeat_ngram_size=2)
logp = sequence_logprob(model, output_beam_with_ngram, input_len=len(input_ids[0]))
print(tokenizer.decode(output_beam_with_ngram[0]))
print(f"\nlog-prob: {logp: .2f}")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In a shocking finding, scientist descovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. Even more surprising to the researchers was the fact that the unicorns spoke perfect the English.


The discovery was made by a team of scientists from the University of California, Davis, and the Universidad Nacional Autónoma de México (UNAM) in Mexico City.

According to a press release, the scientists were conducting a survey of the area when they came across the unicorn herd. They were surprised to find that they were able to communicate with the animals.

log-prob: -77.18


## サンプリング手法

$$P(y_t | y_{<t}, \boldsymbol{x}) = softmax(z_{t,i}) = \frac{exp(z_{t,i})}{\sum^{|V|}_{j=1} exp(z_{t,i})}$$
+ t: 系列時刻
+ $|V|$: 語彙濃度(cardinality)

### 温度パラメータTの導入による出力選択性の制御
+ ソフトマックスを適用する直前に対数を再スケールする温度パラメータTを導入する
+ 出力の多様性を用意に制御できる = Tの調整により出力の確率分布を制御できる
+ $T<<1$ : 分布は原点付近でピークとなり、出現頻度の低い語彙を抑制する = 多様性が減る = コンテキストの一貫性が高まる
+ $T>>1$ : 分布は平坦になり、各語彙(トークン)の出現頻度は同程度になる = 多様性が増える = コンテキストの一貫性が低下する

$$P(y_t | y_{<t}, \boldsymbol{x}) = \frac{exp(z_{t,i}) / T}{\sum^{|V|}_{j=1} exp(z_{t,i} / T)}$$

T=2 : コンテキストが鈍る出力

In [15]:
# generate()関数にtemperatureパラメータを設定して、T=2で出力語彙(トークン)をサンプリングする = コンテキストが鈍る
output_temp = model.generate(input_ids, max_length=max_length, do_sample=True, temperature=2.0, top_k=0)
print(tokenizer.decode(output_temp[0]))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In a shocking finding, scientist descovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. Even more surprising to the researchers was the fact that the unicorns spoke perfect the English.


News amount Janet 2011JuFu lynar 1965Kim slickPLIED pacittilateral Dialogue roundshaped brunt Pugifies EconomippOk Highrazy sn Votersizes Spiritualsolete Further================================================================Place tragedy Some da Soldiers memo marked enrol Programming match contributtable celebrate buttons"Definition...SiThompson dizzategor LevelsDirect diaper 960pacjsoncoçoz Gins two by arrangement Kurds IzzyHKrencieseredith concentrated Compet microbiota lesbians insult BARTG


T=0.5 : コンテキストが固まる出力

In [16]:
output_temp = model.generate(input_ids, max_length=max_length, do_sample=True, temperature=0.5, top_k=0)
print(tokenizer.decode(output_temp[0]))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In a shocking finding, scientist descovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. Even more surprising to the researchers was the fact that the unicorns spoke perfect the English.


The researchers, led by a team of researchers from the University of California, Los Angeles, have named the herd of unicorns "Otomí."


The researchers were led by Dr. Kate Spry from the University of California, Los Angeles, and led by Dr. Daniel Ksepka from the University of California, Santa Cruz.


The researchers have been studying the


### Top-kサンプリングとTop-pサンプリング
+ 温度パラメータの代替or併用
+ 各時刻でサンプリングする語彙(トークン)の種類を制限する

### Top-kサンプリング
+ 確率の高いkトークンだけからサンプリングすることで、確率の低いトークンのサンプリングを避ける

In [17]:
# generate()関数のパラメータtok_kを用いてTop-Kサンプリングを行う
output_topk = model.generate(input_ids, max_length=max_length, do_sample=True, top_k=50)
print(tokenizer.decode(output_topk[0]))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In a shocking finding, scientist descovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. Even more surprising to the researchers was the fact that the unicorns spoke perfect the English.


In order to prove their existence and learn more about them, the researchers travelled to the village of N'Dijor, in the Sierra de la Sierra de San Carlos, Bolivia - a remote valley, and had a closer look into the area without any idea of what they will find inside. After walking for 7 hours and more, they found a herd of unicorns living on a mountain with


### Top-pサンプリング
+ 動的なしきい値(確率しきい値)を使う
+ トークンの候補を選ぶ条件を指定する : 条件「候補語彙(トークン)の各確率の和がある一定確率以上になる」
+ 確率和が一定以上になった候補語彙(トークン)の中からサンプリングする

In [18]:
# generate()関数のパラメータtop_pを用いてTop-pサンプリングを行う
output_topp = model.generate(input_ids, max_length=max_length, do_sample=True, top_p=0.9)
print(tokenizer.decode(output_topp[0]))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In a shocking finding, scientist descovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. Even more surprising to the researchers was the fact that the unicorns spoke perfect the English.


"It is amazing how the unicorns communicated with each other, but only in English, even when they didn't have any other language," study author, Dr. Mario Ponce, a biologist from the University of Quilmes, told the New York Post. "It is the first time anyone has ever found a herd of unicorns in the Andes."


The scientists began studying


Top-kサンプリングとTop-pサンプリングを組み合わせる
+ Top-kの語彙集(プール)からTop-p確率の範囲の語彙分布からサンプリングする

In [19]:
output_topk_topp = model.generate(input_ids, max_length=max_length, do_sample=True, top_p=0.9, top_k=50)
print(tokenizer.decode(output_topk_topp[0]))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In a shocking finding, scientist descovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. Even more surprising to the researchers was the fact that the unicorns spoke perfect the English.


Culturally, the ancient Andes people were obsessed with the mythical creature. The legend goes that one day, a lone wanderer, dressed in a long cloak, climbed a mountain. He took a shortcut by following an old trail to a small lake, in which he fell in love with a mysterious creature, a woman with beautiful hair. At some point, the man's love became a


温度パラメータ/Top-Kサンプリング/Top-pサンプリング

In [23]:
output_mix1 = model.generate(input_ids, max_length=max_length, do_sample=True, temperature=1.0, top_k=50, top_p=0.9)
print(tokenizer.decode(output_mix1[0]))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In a shocking finding, scientist descovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. Even more surprising to the researchers was the fact that the unicorns spoke perfect the English.


The discovery was made by Carlos Pardo at the Center for the Study of Animals, Mammals, and Their Environment, at the University of Oviedo.


Pardo discovered the unicorns on the slopes of the Cerro Negro mountain, while on a hiking trip to study the herds of the Andes, as well as the local mountain inhabitants. As it turned out, one of


温度パラメータ/ビームサーチ/Nグラムペラルティ
+ GPT-2のデフォルト出力に近い

In [25]:
output_mix2 = model.generate(input_ids, max_length=max_length, do_sample=True, temperature=1.0, num_beams=5, no_repeat_ngram_size=2)
print(tokenizer.decode(output_mix2[0]))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In a shocking finding, scientist descovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. Even more surprising to the researchers was the fact that the unicorns spoke perfect the English.


The discovery was made by a team of scientists from the University of California, Santa Cruz, and the National Geographic Society.

According to a press release, the team was conducting a survey of the region when they came across the unicorn herd. The researchers were shocked to discover that they were able to communicate with the animals, which they dubbed "unicorns" because of their resemblance to real-


### デコード戦略のまとめ
+ 最適な普遍的手法は存在しない。
+ タスクにあわせたパラメータを選ぶ。

およその基準
+ 質問タスク -> 温度パラメータ、ビームサーチ、Nグラムペナルティ
+ 文章生成タスク -> Top-k, Top-p